# DiffSynth 원본 Anima · ComfyUI 비교용 기록

Colab **T4 GPU**에서 순서대로 실행합니다. 설정은 저장소의 JSON, 설치·생성·기록은 Python 파일을 사용합니다.

- 생성 HP: `config/generation.json`
- Python·torch·CUDA 빌드 기준: `config/environment.json`
- 모델 파일·revision: `config/model_source.json`
- 원본 실행 옵션·토크나이저: `diffsynth/original_runtime.json`
- 원본 소스: `diffsynth/upstream/` (`source_manifest.json`의 고정 커밋)
- 실행: `diffsynth/run_original.py`, 관찰: `diffsynth/trace_original.py`

원본 tokenizer·Qwen·DiT·VAE·CFG·Euler 연산을 사용합니다. 기록 함수는 원본 입력과 반환값을 그대로 전달합니다.
원본 시간표·난수 생성·FP16 계산을 사용하므로 기존 `matched_euler`와의 차이도 관찰 대상입니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, subprocess, sys
from IPython.display import Image, display

REPOSITORY = 'https://github.com/HisameOgasahara/DiffFlowDiT_test.git'
REVISION = 'main'
PROJECT = Path('/content/DiffFlowDiT_original')
DATA = Path('/content/diffsynth_original_data')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REVISION, REPOSITORY, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], check=True)
DATA.mkdir(parents=True, exist_ok=True)

# 설치·추론의 화면 출력과 로그 파일을 함께 기록합니다.
def launch(command, log_path):
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen([str(x) for x in command], cwd=PROJECT,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            if process.wait():
                raise RuntimeError(f'실행 실패: {log_path}')

## 환경 설치

기존 환경 JSON의 Python 3.13.15·torch 2.14.0+cu130·torchvision 0.29.0+cu130을 사용합니다.
전용 가상환경은 Colab 패키지를 상속하지 않습니다. 기존 ComfyUI용 설치 패키지는 추가하지 않습니다.

In [ ]:
INSTALL_LOG = DATA / 'install.log'
launch([sys.executable, '-u', PROJECT / 'tools/setup_diffsynth_original.py'], INSTALL_LOG)
PYTHON = PROJECT / '.venv-diffsynth-original/bin/python'

## 설정과 모델 준비

저장소 JSON을 데이터 폴더로 복사하여 사용합니다. 아래 `hp`와 `runtime`을 수정할 수 있습니다.
T5는 기존 노트북과 같은 저장소 내 파일을 사용합니다. Qwen과 모델 가중치는 공개 저장소에서 받습니다. Colab 비밀 조회는 하지 않습니다.

In [ ]:
CONFIG = DATA / 'generation.json'
RUNTIME = DATA / 'runtime.json'
for source, target in [(PROJECT / 'config/generation.json', CONFIG),
                       (PROJECT / 'diffsynth/original_runtime.json', RUNTIME)]:
    if not target.exists():
        shutil.copy2(source, target)
hp = json.loads(CONFIG.read_text(encoding='utf-8'))
runtime = json.loads(RUNTIME.read_text(encoding='utf-8'))
# 이전 버전이 저장한 인증 필요 T5 경로만 새 기본 경로로 갱신합니다.
t5 = runtime['tokenizers']['t5']
if t5.get('repository') == 'stabilityai/stable-diffusion-3.5-large':
    defaults = json.loads((PROJECT / 'diffsynth/original_runtime.json').read_text(encoding='utf-8'))
    runtime['tokenizers']['t5'] = defaults['tokenizers']['t5']
# 수정 예: hp['steps'] = 30
CONFIG.write_text(json.dumps(hp, ensure_ascii=False, indent=2), encoding='utf-8')
RUNTIME.write_text(json.dumps(runtime, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'generation': hp, 'runtime': runtime}, ensure_ascii=False, indent=2))
TRACE = 'selected'  # 첫·둘째·마지막 스텝; 시간 측정만 할 때 'none'

In [ ]:
MODELS = DATA / 'models'
launch([PYTHON, '-u', PROJECT / 'diffsynth/prepare_original.py',
        '--config', CONFIG, '--runtime', RUNTIME, '--models', MODELS], DATA / 'prepare.log')

## 생성과 기록

`image.png`, `run.log`, 설정·모델 해시·시간표·환경·시간·메모리·호출 횟수를 저장합니다.
`TRACE='selected'`이면 초기 노이즈, 실제 토큰·마스크·조건, 텍스트 어댑터 출력,
positive/negative DiT 입력·예측, CFG 후 예측, 갱신 전후 latent, VAE 입력·출력을 저장합니다.

비교용 latent 파일은 값 변경 없이 `B,C,T,H,W` 축으로 저장합니다.
`denoised`는 관찰값에서 계산한 FP32 진단값이며 추론에 사용하지 않습니다.
`pixels`는 실제 저장 이미지의 0~1 값이고 양자화 전 VAE 출력은 `decoded_pixels`입니다.
기록에는 CPU 복사·저장 비용이 포함됩니다.

In [ ]:
RUN_NAME = 'diffsynth_original_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT = DATA / 'runs' / RUN_NAME
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
RUN_LOG = DATA / (RUN_NAME + '.log')
try:
    launch([PYTHON, '-u', PROJECT / 'diffsynth/run_original.py', '--config', CONFIG,
            '--runtime', RUNTIME, '--models', MODELS, '--output', OUTPUT,
            '--mode', 'native', '--trace', TRACE], RUN_LOG)
finally:
    if OUTPUT.exists():
        shutil.copy2(RUN_LOG, OUTPUT / 'run.log')
if (OUTPUT / 'image.png').exists():
    display(Image(filename=str(OUTPUT / 'image.png')))
print('실행 폴더:', OUTPUT)

## 결과 이동과 ComfyUI 비교

기존 노트북과 같은 ZIP·비교기를 사용합니다. ComfyUI 결과 ZIP을 업로드하고 아래 `SELECTED`에 두 실행 폴더를 지정합니다.
원본 DiffSynth와 기존 ComfyUI의 실제 sampler·시간표·토크나이저·정밀도 차이를 함께 확인합니다.
조건 길이나 배치 크기가 다르면 비교기는 `shape_mismatch`로 보고하며 값을 임의로 자르지 않습니다.

In [ ]:
# 결과 다운로드 — 필요할 때 True로 변경
DOWNLOAD = False
if DOWNLOAD:
    from google.colab import files
    archive = shutil.make_archive(str(DATA / RUN_NAME), 'zip', OUTPUT.parent, OUTPUT.name)
    files.download(archive)

In [ ]:
# 이전 실행 결과 업로드 — 필요할 때 True로 변경
UPLOAD = False
if UPLOAD:
    from google.colab import files
    import io, zipfile
    for name, payload in files.upload().items():
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            destination = (DATA / 'runs').resolve()
            for item in archive.infolist():
                target = (destination / item.filename).resolve()
                if not target.is_relative_to(destination) or target.exists():
                    raise ValueError(f'허용되지 않거나 이미 존재하는 결과 경로: {item.filename}')
            archive.extractall(destination)

In [ ]:
# 7. 비교할 실행 폴더 선택
RUNS = DATA / 'runs'
available = sorted(p for p in RUNS.iterdir() if (p / 'metrics.json').exists())
for path in available:
    print(path.name)
SELECTED = []  # 예: ['comfyui_matched_euler_...', 'diffusers_matched_euler_...']
if len(SELECTED) >= 2:
    COMPARE = DATA / 'comparisons' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    subprocess.run([str(PYTHON), '-m', 'common.compare', *[str(RUNS / name) for name in SELECTED], '--output', str(COMPARE)], cwd=PROJECT, check=True)
    if (COMPARE / 'comparison.png').exists():
        display(Image(filename=str(COMPARE / 'comparison.png')))
    print((COMPARE / 'comparison.md').read_text(encoding='utf-8'))
else:
    print('다른 노트북도 실행한 뒤 SELECTED에 비교할 폴더 이름을 넣으세요.')